# Production Inference Pipeline

This notebook demonstrates how the trained ticket priority prediction model is used in production.

The pipeline performs the following steps:

- Load the trained machine learning artifacts
- Preprocess incoming ticket data
- Generate TF-IDF features
- Encode metadata
- Predict ticket priority
- Convert predictions back to human-readable labels

In [1]:
import joblib
import pandas as pd

from pathlib import Path
from scipy.sparse import hstack

## Load Production Artifacts

In [2]:
MODELS_DIR = Path("../artifacts")

In [3]:
model = joblib.load(
    MODELS_DIR / "random_forest.pkl"
)

tfidf = joblib.load(
    MODELS_DIR / "tfidf_vectorizer.pkl"
)

priority_encoder = joblib.load(
    MODELS_DIR / "priority_encoder.pkl"
)

In [5]:
metadata_feature_names = joblib.load(
    MODELS_DIR / "metadata_feature_names.pkl"
)

print(len(metadata_feature_names))

988


## Create Sample Production Ticket

Simulate a new support ticket received from a customer.

In [4]:
ticket = {
    "type": "Incident",
    "queue": "Technical Support",
    "tag_1": "Security",
    "tag_2": "Outage",
    "tag_3": "Disruption",
    "tag_4": "Data Breach",
    "text": """
Multiple production servers are inaccessible after a suspected cyber attack.
Several users are unable to log in and business operations have stopped.
Immediate assistance is required.
"""
}

ticket

{'type': 'Incident',
 'queue': 'Technical Support',
 'tag_1': 'Security',
 'tag_2': 'Outage',
 'tag_3': 'Disruption',
 'tag_4': 'Data Breach',
 'text': '\nMultiple production servers are inaccessible after a suspected cyber attack.\nSeveral users are unable to log in and business operations have stopped.\nImmediate assistance is required.\n'}

## Convert Incoming Ticket

The incoming ticket is converted into a DataFrame before applying the production preprocessing pipeline.

In [6]:
ticket_df = pd.DataFrame([ticket])

ticket_df

,type,queue,tag_1,tag_2,tag_3,tag_4,text
0,Incident,Technical Support,Security,Outage,Disruption,Data Breach,\nMultiple production servers are inaccessible...


## Encode Metadata

Convert the categorical metadata into the same one-hot encoded format used during model training.

In [7]:
metadata_columns = [
    "type",
    "queue",
    "tag_1",
    "tag_2",
    "tag_3",
    "tag_4"
]

metadata = pd.get_dummies(
    ticket_df[metadata_columns],
    dtype=int
)

In [9]:
metadata = metadata.reindex(
    columns=metadata_feature_names,
    fill_value=0
)

print(metadata.shape)

(1, 988)


## Transform Ticket Text

Convert the ticket text into TF-IDF features using the trained vectorizer.

In [10]:
text_features = tfidf.transform(
    ticket_df["text"]
)

print(text_features.shape)

(1, 5000)


## Combine Features

The TF-IDF text features and metadata features are combined into a single feature vector matching the format used during model training.

In [11]:
X = hstack([
    text_features,
    metadata.values
])

print(X.shape)

(1, 5988)


## Predict Ticket Priority

In [12]:
prediction = model.predict(X)

prediction

array([0])

## Decode Prediction

Convert the predicted numerical class back into the original ticket priority label.

In [13]:
predicted_priority = priority_encoder.inverse_transform(prediction)

print("Predicted Priority:", predicted_priority[0])

Predicted Priority: high


## Create Reusable Prediction Function

Wrap the complete preprocessing and prediction workflow into a reusable function for deployment.

In [14]:
def predict_priority(
    text,
    ticket_type,
    queue,
    tag_1,
    tag_2="Unknown",
    tag_3="Unknown",
    tag_4="Unknown"
):
    ticket = pd.DataFrame([{
        "type": ticket_type,
        "queue": queue,
        "tag_1": tag_1,
        "tag_2": tag_2,
        "tag_3": tag_3,
        "tag_4": tag_4,
        "text": text
    }])

    metadata = pd.get_dummies(
        ticket[metadata_columns],
        dtype=int
    )

    metadata = metadata.reindex(
        columns=metadata_feature_names,
        fill_value=0
    )

    text_features = tfidf.transform(ticket["text"])

    X = hstack([
        text_features,
        metadata.values
    ])

    prediction = model.predict(X)

    return priority_encoder.inverse_transform(prediction)[0]

In [15]:
predict_priority(
    text="""
Database server has crashed.
Users cannot access the application.
Business operations are stopped.
Immediate assistance required.
""",
    ticket_type="Incident",
    queue="Technical Support",
    tag_1="Database",
    tag_2="Outage",
    tag_3="Infrastructure",
    tag_4="Failure"
)

'high'